# Grondwaterstand SWF
Met dit notebook kan de gemiddelde grondwaterstand bepaald worden per buurt, en per welstandsgebied. Deze wordt dan ook vergeleken met de voorspelling grondwaterstand 2050 hoog. De datafiles zijn van toepassing op de gemeente Súdwest-Fryslân.

#### Setup
Samen met dit notebook zouden de volgende bestanden verschaft moeten zijn:
```
gemeentegrens_2018.gpkg
cbs_buurten.gpkg
welstandsgebieden.gpkg
SWF_Gemiddelde Laagste Grondwaterstand 2050 Hoog.tif
SWF_Gemiddelde Laagste Grondwaterstand Huidig.tif
SWF-GWS.yml
```
De eerste 5 zijn datasets die gebruikt worden in de analyze. De laaste is de .yml file die door conda gebruikt kan worden voor het installeren van een environment met alle packages nodig om dit script te runnen. Gebruik het console command ```conda env create -f SWF-GWS.yml``` daarvoor. Dit gaat er natuurlijk van uit dat je zelf al anaconda of miniconda op je pc hebt geïnstalleerd.

In [ ]:
import xarray as xr
import rioxarray
import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
import pyogrio
from pathlib import Path
import os
import time
import datetime
import numpy as np

### Filepaths
Voor het vormen en bewerken van filepaths wordt [de `pathlib` library](https://docs.python.org/3/library/pathlib.html) gebruikt, dit is een opvolger voor de `os.path` module. Pathlib biedt meer uniforme support tussen de filesystems van verschillende operating systems. Huidige code gaat uit van de current working directory, indien de data in een andere folders staat gebruik dan: ```folderpath = Path(r'schijf_letter:\parent_folder\...\end_folder')``` voor windows.

In [ ]:
# vind alle geotiffs en geopackages
folderpath = Path.cwd() #haalt de huidige folder op waarin jupyter aan het werk is.
GWS_huidig_filepath = folderpath / "SWF_Gemiddelde Laagste Grondwaterstand Huidig.tif"
GWS_2050_filepath = folderpath / "SWF_Gemiddelde Laagste Grondwaterstand 2050 Hoog.tif"
gemeentegrens_filepath = folderpath / "gemeentegrens_2018.gpkg"
buurten_filepath = folderpath / "cbs_buurten.gpkg"
welstandsgebieden_L0_bodemsoorten_L1_filepath = folderpath / "welstandsgebieden.gpkg"
print(GWS_huidig_filepath,"\n",GWS_2050_filepath,"\n",gemeentegrens_filepath,"\n",buurten_filepath,"\n", welstandsgebieden_L0_bodemsoorten_L1_filepath)

Controleer of alle benodigde bestanden aanwezig zijn.

### Raster data: meetingen grondwaterstand

Voor het openen, bewerken en opslaan van raster data bestanden wordt [de library `xarray`](https://docs.xarray.dev/en/stable/index.html) gebruikt. Deze library heeft verschillende backend dependencies zoals GDAL, numpy en rioxarray.

In [ ]:
# open grondwaterstand huidig en print wat details
GWS_ds = xr.open_dataset(GWS_huidig_filepath, decode_times=True)
GWS_ds = GWS_ds.squeeze("band", drop=True)
GWS_ds = GWS_ds.rename({"band_data":"GWS huidig"})

print(GWS_ds)
print(GWS_ds.rio.crs)
print(GWS_ds.rio.bounds())
print(GWS_ds.attrs)
print(GWS_ds.dtypes)

In [ ]:
GWS_ds["GWS huidig"][80:90,90:100].data

Als er metadata in het bestand aanwezig is dan is die niet opgeslagen op de manier zoals xarray het had verwacht, gezien de attributen variable van het object leeg is zoals te zien is in de bovenstaande regel.

In [ ]:
#open grondwaterstand hoog en print wat details
GWS_hoog = xr.open_dataset(GWS_2050_filepath)
GWS_hoog = GWS_hoog.squeeze("band", drop=True)
GWS_hoog = GWS_hoog.rename({"band_data":"GWS hoog"})
print(GWS_hoog)
print(GWS_hoog.rio.crs)
print(GWS_hoog.rio.bounds())
print(GWS_hoog.dtypes)

In [ ]:
#stop beide dataset in één xarray object en bekijk resultaat
GWS_ds = GWS_ds.merge(GWS_hoog)
print(GWS_ds)
print(GWS_ds.dtypes)

### Vectordata: gebied_SWF
Voor het openen, bewerken en opslaan van vectordata wordt [de library `geopandas`](https://geopandas.org/en/stable/getting_started/introduction.html) gebruikt. Ook deze library heeft backend dependencies zoals pandas, numpy, pyogrio, fiona, GDAL en matplotlib.
Geopandas kan maar één dataframe per laag tegelijk maken. Voor geopackages met meerdere lagen zou je een dictionary kunnen maken waarbij je elke laag als aparte entry inleest.

```python
# lees alle lagen en sla elk op als een aparte entry in dictionary
gebied_layers = gpd.list_layers(vectorfiles[0])["name"]
gebied_dfs_dict = {}
for layer in gebied_layers:
    new_data = gpd.read_file(vectorfiles[0], layer = layer)
    gebied_dfs_dict[layer] = new_data

#selecteer en inspecteer één van de lagen
gebied_df_relevant = gebied_dfs_dict[gebied_layers[0]]
gebied_df_relevant.head()
```

In [ ]:
#check hoeveel layers
gebied_layers = gpd.list_layers(gemeentegrens_filepath)
print(gebied_layers)

In [ ]:
gebied_SWF = gpd.read_file(gemeentegrens_filepath)
gebied_SWF.plot()

In [ ]:
#check Coordinate Reference System
gebied_SWF.crs

### Waterstand meetingen volledige gemeente
Selecteer de rasterdata van de waterstand op het gebied van de volledige gemeente.

In [ ]:
# als crs van raster data en vector data overeenkomen, 
# selecteer dan alle rasterdata die binnen Súdwesf-Fryslân valt
assert gebied_SWF.crs == GWS_ds.rio.crs
GWS_SWF_ds = GWS_ds.rio.clip(geometries = gebied_SWF.geometry, crs = GWS_ds.rio.crs)

# plot vectordata van het resultaat
figure, axis = plt.subplots(1,2,figsize=(15,5))
axis[0].axis("off")
axis[1].axis("off")
GWS_SWF_ds["GWS huidig"].plot(ax = axis[0], cmap="BrBG_r")
GWS_SWF_ds["GWS hoog"].plot(ax = axis[1], cmap="BrBG_r")
axis[0].set_title("grondwaterstand huidig")
axis[1].set_title("grondwaterstand 2050 hoog");

### Functie gemiddelde waarde per gebied
Met onderstaande functie kan de gemiddelde waarde per gebied worden berekend.

In [ ]:
def gemiddelde_per_gebied(raster_data_ds,gebieden_df):
    """Bereken de gemiddelde waarde(s) per gebied.
    
    Neemt een xarray dataset met rasterdata van één of meerdere meeteenheden, 
    en een geopandas dataframe met vectordata van één of meerdere gebieden 
    (die binnen het gebied van de rasterdata vallen). 
    Vervolgens wordt de gemiddelde waarde per gebied berekend voor elke meeteenheid.

    parameters:
    raster_data_ds: xarray dataset
    gebieden_df: geopandas dataframe

    returns:
    een dataframe gelijk aan gebieden_df, met een extra kolom voor elke meeteenheid.

    resultaat_df: geopandas dataframe
    """
    assert raster_data_ds.rio.crs == gebieden_df.crs
    resultaat_df = gebieden_df.copy()
    data_labels = list(raster_data_ds.data_vars.keys())
    for label in data_labels:
        resultaat_df[label] = pd.Series()
        
    for index_pos, huidig_gebied in gebieden_df.iterrows():
        try:
            subset = raster_data_ds.rio.clip(geometries = [huidig_gebied['geometry']], crs = gebieden_df.crs, all_touched=True)
            for label in data_labels:
                gemiddelde = subset.mean(skipna = True)[label].item()
                if not pd.isna(gemiddelde):
                    resultaat_df.loc[index_pos, label] = gemiddelde
                else:
                    resultaat_df.drop(index = index_pos, inplace = True)
                    break
        except:
            resultaat_df.drop(index = index_pos, inplace = True)
            
    return resultaat_df

In plaats van gemiddelde waarde kunnen er ook makkelijk andere statistische functies worden gebruikt, zoals:
```python
subset.max()
subset.min()
subset.mean()
subset.median()
subset.prod()
subset.sum()
subset.std()
subset.var()
subset.cumsum()
subset.cumprod()
```

### Waterstand per buurt
Bepaal de gemiddelde waterstand per buurt.

In [ ]:
#controleer hoeveelheid lagen in cbs_buurten
buurt_layers = gpd.list_layers(buurten_filepath)
buurt_layers

Met maar één laag in het bestand is er geen verdere moeilijkheid voor het inlezen van de data

In [ ]:
# lees cbs_buurten en verwijder uit de data de twee die niet relevant zijn
buurten_df = gpd.read_file(buurten_filepath)
drop_index = buurten_df.loc[buurten_df['BU_CODE'].isin(['BU19009997' , 'BU19009998'])].index # maaken een index die alle ongewenste rijen flagt als True
buurten_df = buurten_df.drop(drop_index) #drop verwijdert hele rijen incl. indexlabel, dit kan een sprong in de index achterlaten
buurten_df = buurten_df.reset_index(drop = True) #indien de index simpelweg een oplopende reeks is, dan kan je hiermee de sprongen weghalen, en dus de reeks herstellen
buurten_df.tail() # inspecteer de data

Twee buurtcodes waren niet relevant voor de analyze en zijn verwijderd uit de dataset.

In [ ]:
# plot map van alle buurten
ax = buurten_df.plot()
ax.axis("off");

In [ ]:
# bereken gemiddelde grondwaterstand waarde per buurt
buurten_GWS_df = gemiddelde_per_gebied(GWS_SWF_ds,buurten_df)
print(buurten_GWS_df[['GWS huidig','GWS hoog']].head())#controleer resultaat

Voor elke buurt is de gemiddelde grondwaterstand 'huidig' en 'hoog' berekend, en opgeslagen in het dataframe `buurten_GWS_df`, dat ook alle geometrie voor alle buurten bevat.

In [ ]:
# inspecteer datatypes
buurten_GWS_df.dtypes

Voor de meeste kolommen is het datatype object, voor veel daarvan is een meer geschikt datatype aanwezig.

In [ ]:
# inspecteer dataframe om te kijken welke datatypes geschikt zijn
buurten_GWS_df.head()

In [ ]:
# met alleen maar land in het dataframe is de kolom 'IsWater' niet meer relevant. alternatief zou dit naar een boolean omgezet kunnen worden.
if "IsWater" in buurten_GWS_df.columns:
    buurten_GWS_df.drop(columns="IsWater", inplace=True)
    
# de rest van de datatypes wordt omgezet naar string of float32 afhankelijk van de opgegeven dictionary
buurten_GWS_df = buurten_GWS_df.astype({'BU_CODE':str,'BU_NAAM':str,'WK_CODE':str,'GWS huidig':float,'GWS hoog':float})

In [ ]:
buurten_GWS_df.dtypes

Nu zijn de datatypes van het dataframe beter gepast.

### Waterstand per welvaardsgebied
Bepaal de gemiddelde waterstand per welvaardsgebied.

In [ ]:
welstand_layers = gpd.list_layers(welstandsgebieden_L0_bodemsoorten_L1_filepath)
print(welstand_layers)

In [ ]:
welstandsgebieden_df = gpd.read_file(welstandsgebieden_L0_bodemsoorten_L1_filepath, layer = 0)
welstandsgebieden_df.rename(columns={"D_SWF_WELSTANDSGEBIEDEN_SK":"GEBIEDSCODE"},inplace=True)
welstandsgebieden_df.head()

In [ ]:
welstandsgebieden_df.plot()

In [ ]:
# bereken de gemiddelde waardes, en controleer of ze toegevoegd zijn
welstandsgebieden_GWS = gemiddelde_per_gebied(GWS_SWF_ds,welstandsgebieden_df)
welstandsgebieden_GWS.head()

In [ ]:
welstandsgebieden_GWS.plot()

In [ ]:
#controleer datatypes
welstandsgebieden_GWS.dtypes

In [ ]:
#pas datatypes aan, en bereken verschil in GWS huidig en GWS hoog
welstandsgebieden_GWS = welstandsgebieden_GWS.astype({"GEBIEDSCODE":int,"GEBIED":str,"AANDUIDING":str,"GWS huidig":float,"GWS hoog":float})
welstandsgebieden_GWS["GWS verschil"] = welstandsgebieden_GWS["GWS hoog"] - welstandsgebieden_GWS["GWS huidig"]
welstandsgebieden_GWS.head()

### Visualisatie grondwaterstand per buurt en welstandsgebied
Om de analyse te controleren maken we verschillende plots waar de grondwaterstand op weergegeven wordt.

Deze code is in V2 minder relevant: maak plot van grondwaterstand per buurt met 6 verschillende kleurwaarden met bereik gegroepeerd volgens 'NaturalBreaks' schema. 

``` python
ax = buurten_GWS_df.plot(
    column = "GWS huidig",
    edgecolor = "black",
    linewidth = 0.1,
    legend = True,
    cmap = "BrBG_r",
    scheme = 'NaturalBreaks',
    k = 6,
    legend_kwds = {"fmt":"{:.01f}M",
                   "interval":"True",
                   "loc":"upper right", 
                   "bbox_to_anchor":(0.93,0.518,0.5,0.5)}
    );
ax.set_axis_off();
ax.set_title("grondwaterstand diepte per buurt (huidig)");
```
Het bereik van de grondwaterstand wordt opgedeelt in K=6 verschillende groepen, waarbij het bereik van elke groep individueel wordt bepaald volgens het 'natural break' schema. De schemas worden berekend door de mapclassify library. Andere schema opties zijn: ‘box_plot’, ‘equal_interval’, ‘fisher_jenks’, ‘fisher_jenks_sampled’, ‘headtail_breaks’, ‘jenks_caspall’, ‘jenks_caspall_forced’, ‘jenks_caspall_sampled’, ‘max_p_classifier’, ‘maximum_breaks’, ‘natural_breaks’, ‘quantiles’, ‘percentiles’, ‘std_mean’ or ‘user_defined’.

Er wordt een warning over memory leaks gegeven, tot noch toe lijkt er geen probleem te zijn. Zou dit veranderen dan kan men in de toekomst kijken of het limiteren van cpu cores wat de python kernel mag gebruiken het oplost.

In [ ]:
# maak plots met een continue kleurschaal. met bruin voor grondwater diepte boven 0 en blauw voor onder 0.

vmin = buurten_GWS_df["GWS hoog"].min()
vmax = buurten_GWS_df["GWS huidig"].max()
divnorm = mpl.colors.TwoSlopeNorm(vmin = vmin, vmax = vmax, vcenter = 0)
figure, axis = plt.subplots(1,2,figsize=(15,7))

buurten_GWS_df.plot(ax = axis[0],
                    column = "GWS huidig",
                    edgecolor = "black",
                    linewidth = 0.1,
                    #legend = True,
                    cmap = "BrBG_r",
                    norm = divnorm
                   )

buurten_GWS_df.plot(ax = axis[1],
                    column = "GWS hoog",
                    edgecolor = "black",
                    linewidth = 0.1,
                    legend = True,
                    cmap = "BrBG_r",
                    norm = divnorm,
                    legend_kwds = {"ax":axis,
                                   "label":"grondwater diepte t.o.v. maaiveld",
                                   "location":"bottom",
                                   "orientation":"horizontal",
                                   "shrink":0.5,
                                   "ticks":[-0.22,-0.11,0,0.8,1.6,2.4]})
axis[0].axis("off");
axis[1].axis("off");
axis[0].set_title("grondwaterstand diepte per buurt: huidig");
axis[1].set_title("grondwaterstand diepte per buurt: 2050 hoog");

In [ ]:
vmin = welstandsgebieden_GWS["GWS hoog"].min()
vmax = welstandsgebieden_GWS["GWS huidig"].max()
divnorm = mpl.colors.TwoSlopeNorm(vmin = vmin, vmax = vmax, vcenter = 0)
ticks = [round(vmin+0.01,2),round(vmin/2,2),0,round(vmax/3,2),round((vmax*2)/3,2),round(vmax-0.01,2)]
figure, axis = plt.subplots(1,2,figsize=(15,7))

welstandsgebieden_GWS.plot(ax = axis[0],
                           column = "GWS huidig",
                           edgecolor = "black",
                           linewidth = 0.1,
                           cmap = "BrBG_r",
                           norm = divnorm
                   )

welstandsgebieden_GWS.plot(ax = axis[1],
                           column = "GWS hoog",
                           edgecolor = "black",
                           linewidth = 0.1,
                           legend = True,
                           cmap = "BrBG_r",
                           norm = divnorm,
                           legend_kwds = {"ax":axis,
                                          "label":"grondwater diepte t.o.v. maaiveld",
                                          "location":"bottom",
                                          "orientation":"horizontal",
                                          "shrink":0.5,
                                          "ticks":ticks})
axis[0].axis("off");
axis[1].axis("off");
axis[0].set_title("grondwaterstand diepte per welstandsgebied: huidig");
axis[1].set_title("grondwaterstand diepte per welstandsgebied: 2050 hoog");

### Opslaan van de nieuwe data

Het dataframe met de nieuwe data kan natuurlijk in hetzelfde format opgeslagen worden als hoe de oorspronkelijke data is aangeleverd (.gpkg), maar de data kan bijvoorbeeld ook naar een PostGIS database geschreven worden (`geopandas.GeoDataFrame.to_postgis()`).

Verdere supported file formats kunnen gevonden worden met `pyogrio.list_drivers()`. Een output folder wordt aangemaakt om conflicten met het inlezen van aanwezige bestanden te voorkomen in de situatie dat dit script meermaals in eenzelfde folder wordt gebruikt.

```python
def get_times(filepath):
    stats = os.stat(filepath)
    creation_time = time.ctime(stats.st_ctime)
    last_modified_time = time.ctime(stats.st_mtime)
    last_accessed_time = time.ctime(stats.st_atime)
    print(f"creation:{creation_time}\n modified:{last_modified_time}\n accessed:{last_accessed_time}")
    return (creation_time, last_modified_time, last_accessed_time)
```

Bovenstaande functie kan gebruikt worden om tijden uit een specifiek bestand te halen. Hier is dat niet heel relevant.

Wat code die gebruikt kan worden voor het verwerken van timestamps in de data.

```python
# vul de lege tijden in en pas format aan voor output.
welstandsgebieden_GWS["DATE_TO"] = np.datetime64("2050-01-01")
welstandsgebieden_GWS["MODIFICATION"] = np.datetime64("today")
welstandsgebieden_GWS.rename(columns={"GWS hoog":"GWS 2050 hoog"}, inplace = True)
```

In [ ]:
welstandsgebieden_GWS.tail()

In [ ]:
welstandsgebieden_GWS.dtypes

De data is al aanwezig, maar mag eerst nog conform aan het juiste formaat gemaakt worden.

In [ ]:
column_list = ["INDICATOR","GEBIEDSTYPE","GEBIEDSCODE","DATE_FROM","DATE_TO","GWS huidig","GWS hoog"]
welstandsgebieden_GWS["INDICATOR"] = "Gemiddeld Laagste Grondwaterstand"
welstandsgebieden_GWS["GEBIEDSTYPE"] = "d_swf_welstandsgebieden"
welstandsgebieden_GWS[["INDICATOR","GEBIEDSTYPE"]] = welstandsgebieden_GWS[["INDICATOR","GEBIEDSTYPE"]].astype("string")
welstandsgebieden_GWS = welstandsgebieden_GWS.reindex(columns= column_list)

In [ ]:
welstandsgebieden_GWS.tail()

In [ ]:
welstandsgebieden_GWS_huidig_out_df = welstandsgebieden_GWS.drop(columns= "GWS hoog")
welstandsgebieden_GWS_huidig_out_df.rename(columns= {"GWS huidig":"WAARDE"}, inplace=True)
welstandsgebieden_GWS_huidig_out_df["WAARDE"] = round(welstandsgebieden_GWS_huidig_out_df["WAARDE"] * 100)
welstandsgebieden_GWS_huidig_out_df["WAARDE"] = welstandsgebieden_GWS_huidig_out_df["WAARDE"].astype(int)

In [ ]:
welstandsgebieden_GWS_huidig_out_df.tail()

In [ ]:
welstandsgebieden_GWS2050_hoog_out_df = welstandsgebieden_GWS.drop(columns= "GWS huidig")
welstandsgebieden_GWS2050_hoog_out_df.rename(columns={"GWS hoog":"WAARDE"}, inplace=True)
welstandsgebieden_GWS2050_hoog_out_df["INDICATOR"] = "Trend GLG naar 2050"
welstandsgebieden_GWS2050_hoog_out_df["GEBIEDSTYPE"] = "d_swf_welstandsgebieden"
welstandsgebieden_GWS2050_hoog_out_df["INDICATOR"] = welstandsgebieden_GWS2050_hoog_out_df["INDICATOR"].astype("string")
welstandsgebieden_GWS2050_hoog_out_df["GEBIEDSTYPE"] = welstandsgebieden_GWS2050_hoog_out_df["GEBIEDSTYPE"].astype("string")
welstandsgebieden_GWS2050_hoog_out_df["WAARDE"] = round(welstandsgebieden_GWS2050_hoog_out_df["WAARDE"] * 100)
welstandsgebieden_GWS2050_hoog_out_df["WAARDE"] = welstandsgebieden_GWS2050_hoog_out_df["WAARDE"].astype(int)

In [ ]:
welstandsgebieden_GWS2050_hoog_out_df.tail()

Eenmaal in het juiste formaat kan de data naar bestanden uitgeschreven worden. Er is gekozen voor een .csv-bestand, omdat geodata niet meer aanwezig is in de resultaten, en .csv makkelijk werkt met databases.

In [ ]:
# schrijf de nieuwe data uit
outdir = Path("out_dir")
if not Path.exists(outdir):
    outdir.mkdir()
welstandsgebieden_GWS_huidig_out_df.to_csv(path_or_buf= outdir/"GWS_huidig_welstandsgebieden.csv", index=False)
welstandsgebieden_GWS2050_hoog_out_df.to_csv(path_or_buf= outdir/"GWS2050_hoog_welstandsgebieden.csv", index=False)